# nave-xai: NAVE explanations of vision encoders

**NAVE** (Neuro-Activated Vision Explanations) explains a frozen vision encoder: it extracts
concepts from the encoder's feature activations at several depths and shows how the encoder
decomposes an image, before any classifier gets involved.

> Boubekki, Fadel, Mair. *Multi-Depth Concept Extraction for Post-Hoc Vision Encoder Explanation.*
> [arXiv:2411.19700](https://arxiv.org/abs/2411.19700). **If you use nave-xai, please cite the paper.**

The default encoder, ViT-Ti/16 trained on ImageNet, is shipped with the package. Other encoders
download their weights on first use (about 50-100 MB); set `TRAINING = DINO = "random"` to skip that.

In [ ]:
# %pip install nave-xai
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cbook
from matplotlib.image import imread
from nave_xai import NAVEImage

image = imread(cbook.get_sample_data("grace_hopper.jpg", asfileobj=False))   # ships with matplotlib

def show(ax, explainer, exp, title=""):
    """An explanation map in the explainer's concept colours."""
    ax.imshow(explainer.palette[exp.explanation]); ax.set_title(title, fontsize=10); ax.axis("off")

TRAINING = "imagenet"    # "random" to skip downloads
DINO = "dino"            # "random" to skip downloads
plt.imshow(image); plt.axis("off");

## 1. Explain an image

`explain` returns an `ImageExplanation`: `explanation` gives the concept of every pixel. One concept
can appear in several places. Without a previous `fit`, `explain` fits on the image it is given.

In [ ]:
explainer = NAVEImage(training=TRAINING, k=10)            # ViT-Ti/16
exp = explainer.explain(image)
print(exp.explanation.shape, exp.K, "concepts")

fig, axes = plt.subplots(1, 3, figsize=(13, 4.6))
axes[0].imshow(image); axes[0].set_title("image"); axes[0].axis("off")
show(axes[1], explainer, exp, "NAVE explanation")
picture = explainer.palette[exp.explanation].copy()
picture[exp.border(2) == 1] = 255                     # region borders
show(axes[2], explainer, exp, "")
axes[2].imshow(picture); axes[2].set_title("with region borders");

The share of the image each concept covers, and the connected regions (superpixels):

In [ ]:
shares = np.bincount(exp.explanation.ravel(), minlength=exp.K) / exp.explanation.size
plt.figure(figsize=(6, 2.8))
plt.bar(range(exp.K), shares, color=explainer.palette / 255)
plt.xlabel("concept"); plt.ylabel("share of the image")
print(exp.segmentation().max() + 1, "connected regions")

**Optional: uncertainty.** With `uncertainty=True`, every pixel also gets the normalised entropy of its
concept assignment, H(softmax(-d² / T)) / log K, from 0 (certain) to 1.

In [ ]:
u = explainer.explain(image, uncertainty=True).uncertainty
colours = explainer.palette[exp.explanation] / 255
fig, axes = plt.subplots(1, 2, figsize=(9, 4.6))
axes[0].imshow(u, cmap="inferno", vmin=0, vmax=1); axes[0].set_title("uncertainty"); axes[0].axis("off")
axes[1].imshow((1 - u)[..., None] * colours + u[..., None])          # fades to white where uncertain
axes[1].set_title("opacity = certainty"); axes[1].axis("off");

**Everything at once.** `summarize` gathers the image, its explanation (here with certainty as
opacity), the saliency of the predicted class, and every concept's closest patch, with the model's
prediction in the title. With a single image there is no correlation between concepts and classes to
compute, so the row under the explanation shows instead each concept's **mean saliency** in this image:
red concepts support the predicted class, blue ones speak against it. Section 6 shows the same figure
after fitting on several images, with the correlation.

In [ ]:
explainer.summarize(image, uncertainty=True, saliency=True);

## 2. Depth: from edges to semantics

Shallow layers carry detail and edges, deep layers semantics. In the paper's notation, N4 is the
last ResNet block alone and N234 blocks 2 to 4.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4.5))
for ax, layers in zip(axes, ["N4", "N34", "N234"]):
    ex = NAVEImage(model="resnet50", training=TRAINING, layers=layers, k=10)
    show(ax, ex, ex.explain(image), f"ResNet50 {layers}")

## 3. Granularity, boundaries, projectors

`k` sets the number of concepts; `upsample` how concept scores go from the feature grid to pixels
(`"nearest"` gives the paper's blocky maps). `projector="bayesian"` learns how many concepts the
image needs (k is an upper bound); `projector="pca"` uses principal components.

In [ ]:
settings = [dict(k=4), dict(k=20), dict(k=10, upsample="nearest"),
            dict(k=100, projector="bayesian", fit_on="sample", sample_size=5000), dict(k=5, projector="pca")]
fig, axes = plt.subplots(1, 5, figsize=(18, 4.2))
for ax, kw in zip(axes, settings):
    ex = NAVEImage(training=TRAINING, **kw)
    e = ex.explain(image)
    show(ax, ex, e, ", ".join(f"{k}={v}" for k, v in kw.items() if k not in ("sample_size", "fit_on")) + f"\n{len(np.unique(e.explanation))} concepts used")

## 4. Large images

When an image's longer side reaches three times the model's input size (672 px for the built-in
models), it is processed in 448 px windows placed every 224 px; every pixel comes from the window
whose centre is nearest, far from any window edge. Concepts are fitted on all windows together, so
they mean the same everywhere.

In [ ]:
from PIL import Image
big = np.asarray(Image.fromarray(image).resize((1536, 1800)))      # 2.8 megapixels
ex = NAVEImage(training=TRAINING, k=10)
fig, ax = plt.subplots(figsize=(5, 6))
show(ax, ex, ex.explain(big), "2.8 megapixels, in windows")

## 5. Concepts across images

`fit` extracts one set of concepts from many images; `predict` explains any image with them. `fit`
also computes the concept `patches`, the predicted `classes` and the `correlation` of concepts with
them. Here: crops and flips of one photo; use your own dataset instead.

In [ ]:
crops = [image[y:y + 400, x:x + 400] for y, x in [(0, 0), (100, 50), (200, 112), (150, 0)]]
images = crops + [c[:, ::-1] for c in crops]

explainer = NAVEImage(model="resnet50", training=TRAINING, k=8).fit(images)
print("predicted classes:", explainer.classes,
      "| correlation:", "none (needs at least two classes)" if explainer.correlation is None else explainer.correlation.shape)
explanations = explainer.predict(images[:4])
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for i, e in enumerate(explanations):
    axes[0, i].imshow(images[i]); axes[0, i].axis("off")
    show(axes[1, i], explainer, e)

Reordering by class: concepts go under the class they correlate with most, and `recolor=True` gives
each class a colour and its concepts shades of it. The same concepts cover the same regions; only
their numbers and colours change. This needs a correlation, so at least two predicted classes among
the fitted images: crops of a single photo may all get the same class, and then the concepts keep
their order.

In [ ]:
if explainer.correlation is not None:
    explainer.reorder(by="classes", recolor=True)
else:
    print("A single predicted class: no correlation, so no reordering by class.")
explanations = explainer.predict(images[:4])
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for i, e in enumerate(explanations):
    axes[0, i].imshow(images[i]); axes[0, i].axis("off")
    show(axes[1, i], explainer, e)

Every concept's closest and largest cell in the fitting set (`patches`):

In [ ]:
K = len(explainer.patches)
fig, axes = plt.subplots(2, K, figsize=(1.3 * K, 3))
for k in range(K):
    for row, name in enumerate(["closest", "largest"]):
        axes[row, k].imshow(explainer.patches[k, row]); axes[row, k].set_xticks([]); axes[row, k].set_yticks([])
        for spine in axes[row, k].spines.values():
            spine.set_color(explainer.palette[k] / 255); spine.set_linewidth(3)
    axes[0, k].set_title(str(k), fontsize=9)
axes[0, 0].set_ylabel("closest"); axes[1, 0].set_ylabel("largest");

## 6. Summary

`summarize` puts it all together for one image, in the style of the paper's Figure 1:
- **top:** the image, its explanation with the fitted concepts, and the concept-based saliency of the
  predicted class;
- **then, for every block of 10 concepts:** their correlation with the classes seen during `fit` (or,
  when the fitted images had a single class, each concept's mean saliency in this image), and their
  closest cell, framed in the concept's colour.

Here it explains the whole photo with the concepts fitted on the crops above, so the correlation row
describes the fitted crops, and the explanation shows where those concepts appear in the full image.

In [ ]:
explainer.summarize(image, uncertainty=True, saliency=True);

## 7. Groups

With `groups=`, one projector is fitted per group (the paper's protocol: one per class). Concepts are
numbered by group, every group gets a colour family, and `classes` are the groups.

In [ ]:
grouped = NAVEImage(model="resnet50", training=TRAINING, k=4).fit(images, groups=[0, 0, 0, 0, 1, 1, 1, 1])
print(len(grouped.palette), "concepts, classes", grouped.classes)
fig, ax = plt.subplots(figsize=(4, 4))
show(ax, grouped, grouped.predict(images[0]), "two groups of 4 concepts")

## 8. Concept-based saliency

The cosine between the deepest layer's features and the classifier's vector of the predicted class,
averaged over NAVE regions from the deepest layer to the shallowest.

In [ ]:
sal = NAVEImage(model="resnet50", training=TRAINING, layers=[2, 3, 4], k=10).saliency(image)
half = np.abs(sal).max()
plt.imshow(sal, cmap="seismic", vmin=-half, vmax=half); plt.colorbar(); plt.axis("off");

## 9. Your own encoder

Any PyTorch model works (`pip install "nave-xai[torch]"`); layers are module names.

In [ ]:
try:
    import torchvision
    model = torchvision.models.resnet18(weights="DEFAULT")
    ex = NAVEImage(model, layers=["layer3", "layer4"], k=10)
    fig, ax = plt.subplots(figsize=(4, 5))
    show(ax, ex, ex.explain(image), "torchvision ResNet18")
except ImportError:
    print("Install torch and torchvision to run this cell.")

## 10. Time series and text

The same method explains sequence encoders, with one concept per time step or per token:

- **[Time series](series.ipynb):** `NAVESeries` on a 1D ResNet trained on smartwatch data (BasicMotions):
  channels over the segmentation, the concepts' snippets and profiles, their correlation with the
  activities.
- **[Text](text.ipynb):** `NAVEText` with spaCy's encoder and with a topic classifier on Reuters news: the
  document on its concepts' colours, the saliency in the font, the concept table.

Both notebooks say what they download and need (PyTorch, and aeon or NLTK and spaCy).

## Citation

If you use nave-xai, please cite the paper:

```bibtex
@article{boubekki2024nave,
  title   = {Multi-Depth Concept Extraction for Post-Hoc Vision Encoder Explanation},
  author  = {Boubekki, Ahc{\`e}ne and Fadel, Samuel G. and Mair, Sebastian},
  journal = {arXiv preprint arXiv:2411.19700},
  year    = {2024}
}
```